# 00D · 时序场景状态与 Tracking：模型为什么不能只看当前帧？

单帧 perception 给的是 observation，不是稳定的 world state。自动驾驶需要知道：这个目标是不是同一个目标？它的速度和加速度是多少？自车运动造成的相对位移是多少？观测丢失时应该保持多久？

本节不从 Kalman Filter 的公式开始，而从一个工程接口开始：

```text
observation_t + ego_motion_t + previous_state_{t-1}
    → association / state estimation
    → agent_state_t = {id, position, velocity, age, uncertainty}
```

`07` 会把这里的 toy association、dropout、outlier 和 timestamp offset 扩展为可调实验。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(12)
time = np.arange(0, 8, 0.1)
true_position = 8.0 + 1.8 * time + 0.25 * np.sin(time)
observation = true_position + rng.normal(0, 0.45, len(time))
observation[30:40] = np.nan
observation[58] += 2.2

def smooth_track(values, alpha=0.25):
    estimate = []
    current = values[0]
    for value in values:
        if np.isfinite(value):
            current = alpha * value + (1 - alpha) * current
        estimate.append(current)
    return np.asarray(estimate)

estimated = smooth_track(observation)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time, true_position, label="latent actor state", linewidth=2)
ax.scatter(time, observation, s=12, alpha=0.55, label="noisy observation")
ax.plot(time, estimated, label="simple temporal estimate")
ax.axvspan(time[30], time[39], color="orange", alpha=0.15, label="dropout")
ax.legend()
ax.set(xlabel="time / s", ylabel="longitudinal position / m", title="Tracking is state estimation over imperfect observations")


In [ ]:
def tracking_report(dropout_start=3.0, dropout_duration=1.0, outlier_size=2.0, alpha=0.25):
    local = true_position + rng.normal(0, 0.45, len(time))
    dropout = (time >= dropout_start) & (time < dropout_start + dropout_duration)
    local[dropout] = np.nan
    local[np.argmin(np.abs(time - 5.8))] += outlier_size
    estimate = smooth_track(local, alpha=alpha)
    valid = ~dropout
    rmse = np.sqrt(np.mean((estimate[valid] - true_position[valid]) ** 2))
    max_gap = int(dropout.sum())
    print(f"dropout frames={max_gap}, track RMSE outside dropout={rmse:.3f} m")
    print("state fields to preserve: track_id, position, velocity, age, covariance/uncertainty")

from ipywidgets import FloatSlider, interact
interact(
    tracking_report,
    dropout_start=FloatSlider(min=0, max=6, step=0.5, value=3, description="dropout start"),
    dropout_duration=FloatSlider(min=0, max=2.5, step=0.1, value=1, description="duration / s"),
    outlier_size=FloatSlider(min=0, max=5, step=0.25, value=2, description="outlier / m"),
    alpha=FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25, description="update alpha"),
)


## 领域检查点

- tracking 的“记忆”为什么会把错误传播到 prediction？
- dropout 期间是继续输出预测、降低置信度、冻结轨迹，还是触发降级？谁负责这个决定？
- ego-motion compensation 和 actor velocity estimation 各自需要什么输入？

**下一步**：`07` 深入 tracking；`08` 使用 agent state 评估未来轨迹。不要在没有定义 state 和 timestamp 的情况下直接讨论 prediction model。
